# 遠征前壓力測試：什麼樣的資料會讓固定的 Flow Matching framework 開始做不好？

課堂實作裡，同一個 `FlowMatching` class 可以生成影像，也可以生成 action chunks。看起來只要換資料與 neural network 的 input/output shape，事情就解決了。

但真實世界的資料不會永遠像課堂 toy 那麼友善：有些 modes 很少、有些答案擠得很近、有些成功路徑非常窄，有些 observations 則沒有把真正重要的資訊全部說出來。

所以這份作業要問的是：

> **在不修改 Flow Matching 演算法的前提下，什麼樣合理的 continuous data problem，會讓課堂上的固定 framework 開始做不好？我們怎麼知道原因是什麼？**

這是一份兩週作業。兩週內請自己安排時間；**沒有中間 checkpoint**。最後一次繳交：

1. 完成作答與實驗的 `.ipynb`；
2. 同一份 notebook 的 PDF 或 HTML export；
3. 一張 summary figure；
4. AI／research decision log。

這份作業不要求你把模型救回來。沒有成功讓它完全失效，但問題合理、預測清楚、實驗能推翻自己、分析誠實，會比做出很大的 failure 卻說不清楚原因得到更高的分數。


## 評分方式

| 項目 | 配分 |
| --- | ---: |
| 真實問題與 toy 的合理連結 | 20 |
| 數學定義、notation 與 symbol-to-code | 20 |
| Failure intuition 與可被推翻的預測 | 15 |
| 控制變因、truth reference、結果與 useful value | 25 |
| 結果符合／不符合預期的解釋 | 15 |
| Reproducibility 與 AI-use appendix | 5 |

數字好不好看只佔「控制變因、truth reference、結果與 useful value」的一部分。真正要評的是一條可以被檢查的推理：

$$
\text{real problem}
\rightarrow \text{controlled toy}
\rightarrow \text{failure intuition}
\rightarrow \text{prediction}
\rightarrow \text{experiment}
\rightarrow \text{explanation}.
$$

有三個 validity gates，不是額外加分項：$\lambda$ 必須真的改變資料分布、easy baseline 必須相對 truth reference 可用、主要 metric 必須有 truth-vs-truth 或 Bayes/oracle calibration。若其中一項沒有處理，漂亮的 hard failure 也不能直接支持你的結論。


---

# 0. 先把不能改的東西固定下來

這份作業的自由度在 **data problem**，不在 Flow Matching algorithm。所有正式實驗都固定：

$$
T\sim U[0,1],
\qquad
X_0\sim\mathcal N(0,I),
\qquad
(X_1,C)\sim p_{\mathrm{data}}.
$$

$X_0$ 與 $(X_1,C)$ 獨立抽樣；$X_1$ 與 $C$ 則必須來自同一筆資料。不能把某一筆 observation 和另一筆資料的 target 隨便配在一起。

Probability path 與 training target 固定為

$$
I_t=(1-t)X_0+tX_1,
\qquad
U_t=X_1-X_0.
$$

Training objective 固定為

$$
\mathcal L(\theta)
=
\mathbb E_{T,X_0,(X_1,C)}
\left[
\left\|v_\theta(I_T,T,C)-U_T\right\|^2
\right].
$$

Sampling equation 固定為

$$
\frac{\mathrm dX_t}{\mathrm dt}
=v_\theta(X_t,t,c),
\qquad
X_0\sim\mathcal N(0,I).
$$

如果是 unconditional problem，請令 `C = None`；不要另外發明一個沒有作用的 condition。


## 正式實驗的固定設定

你可以因為資料 shape 改變 network 的 input/output dimension 與 `cond_dim`，但不能改下面的 capacity 與 training/sampling budget。這組 budget 是為了讓課堂裡最基本的 image toy 與 trajectory toy 先有一個可用的 easy baseline；**hard 比 easy 差多少**才是這份作業要研究的量，而不是讓三個 level 一開始就全部做不好。

| 設定 | 固定值 |
| --- | ---: |
| Velocity network | `FlatVelocityNetwork` |
| Hidden width | 512 |
| Hidden layers | 3 |
| Time features | 8 frequencies |
| Batch size | 512 |
| Optimizer | Adam |
| Learning rate | $2\times 10^{-3}$ with cosine decay |
| Training steps | 5,000 |
| Sampler | Heun |
| ODE integration steps | 50 |
| Network evaluations | 100（每個 Heun step 兩次） |
| Default evaluation samples | 1,000 |
| Final training seeds | 101, 202, 303 |

你可以用更少的 steps 與較少 samples 做 smoke test，但不能把 smoke-test 數字當成正式結果。

正式比較前還有一個必要的 **easy-baseline gate**：你的 easy toy 必須先在預先寫下的 task metric 上接近 real-vs-real reference。若 easy 都明顯做不好，不能把 hard 的結果解讀成你設計的 data property 所造成的 failure。這時應該調整 toy 的共同表示方式或 normalization，三個 levels 一起改；不能只救 easy，也不能改上面的 fixed budget。


In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

HIDDEN = 512
LAYERS = 3
N_TIME_FREQ = 8
BATCH_SIZE = 512
LEARNING_RATE = 2e-3
TRAIN_STEPS = 5000
SAMPLE_STEPS = 50
N_EVAL = 1000
FINAL_SEEDS = [101, 202, 303]

def set_seed(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)

print("device:", DEVICE)


## 共用的 Flow Matching code

下面兩個 classes 和課堂實作相同。正式實驗不得修改。

`FlatVelocityNetwork` 是我們刻意固定的簡單 baseline。它不一定是影像或軌跡最好的 architecture；正因為它固定，才有辦法比較不同 data properties 對同一個 framework 的影響。


In [ ]:
def time_features(t, n_freq=N_TIME_FREQ):
    t = t.reshape(-1, 1)
    k = (2 ** torch.arange(n_freq, device=t.device, dtype=t.dtype)) * math.pi
    return torch.cat([t, torch.sin(t * k), torch.cos(t * k)], dim=1)


class FlatVelocityNetwork(nn.Module):
    def __init__(self, sample_shape, cond_dim=0, hidden=HIDDEN, layers=LAYERS,
                 n_freq=N_TIME_FREQ):
        super().__init__()
        self.sample_shape = tuple(sample_shape)
        self.cond_dim = cond_dim
        self.n_freq = n_freq
        data_dim = math.prod(self.sample_shape)
        in_dim = data_dim + (1 + 2 * n_freq) + cond_dim

        blocks = []
        for _ in range(layers):
            blocks += [nn.Linear(in_dim, hidden), nn.SiLU()]
            in_dim = hidden
        blocks.append(nn.Linear(in_dim, data_dim))
        self.net = nn.Sequential(*blocks)

    def forward(self, x, t, c=None):
        batch = len(x)
        pieces = [x.reshape(batch, -1), time_features(t, self.n_freq)]
        if self.cond_dim:
            if c is None:
                raise ValueError("This velocity network requires condition C.")
            pieces.append(c.reshape(batch, self.cond_dim))
        velocity = self.net(torch.cat(pieces, dim=1))
        return velocity.reshape_as(x)


class FlowMatching:
    def __init__(self, velocity, lr=LEARNING_RATE):
        self.velocity = velocity
        self.lr = lr

    def loss(self, x0, x1, c=None):
        batch = len(x0)
        t = torch.rand(batch, device=x0.device)
        t_view = t.reshape(batch, *([1] * (x0.ndim - 1)))
        xt = (1 - t_view) * x0 + t_view * x1
        target_velocity = x1 - x0
        return ((self.velocity(xt, t, c) - target_velocity) ** 2).mean()

    def fit(self, get_batch, steps=TRAIN_STEPS, name="model", verbose=True):
        optimizer = torch.optim.Adam(self.velocity.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)
        history = []
        report_every = max(1, steps // 5)
        for step in range(1, steps + 1):
            loss = self.loss(*get_batch())
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            scheduler.step()
            history.append(loss.item())
            if verbose and (step == 1 or step % report_every == 0):
                print(f"[{name}] step {step:4d}/{steps}: loss = {np.mean(history[-report_every:]):.4f}")
        return history

    @torch.no_grad()
    def sample(self, x0, c=None, steps=SAMPLE_STEPS):
        x = x0.clone()
        dt = 1.0 / steps
        batch = len(x)
        for i in range(steps):
            t0 = torch.full((batch,), i / steps, device=x.device)
            t1 = torch.full((batch,), (i + 1) / steps, device=x.device)
            v0 = self.velocity(x, t0, c)
            proposal = x + dt * v0
            v1 = self.velocity(proposal, t1, c)
            x = x + 0.5 * dt * (v0 + v1)
        return x


---

# 問題一：從真實世界設計一個受控制的 toy

這裡不是要想辦法把模型弄壞。先找一個真實世界可能出現的 data property，再問：**如何把它縮成一個保留核心困難、其他部分又簡單得可以看懂的 toy？**

你可以選擇影像生成，或動作與軌跡 policy learning。兩個 track 的自由度稍微不同：

- **影像組採半開放式設計。**先提出自己關心的 real problem，再把它對應到下面五個 failure families 之一。Family 是數學骨架，不是影像題材；你仍然可以把它長成細胞、衛星建物、道路、材料影像或自己的專題問題。
- **Policy 組維持較開放。**障礙物、窄通道、route 與 task success 本來就比較容易定義，但仍然只能改一個 primary property。

## 影像組：五種研究骨架

每個 image package 會規定三件事：一個 difficulty axis、至少一個 primary metric，以及一個防止假解的 guard metric。課程提供的 starter generator 也會標出一個 **anchor setting**：它的作用是確認這個骨架在固定 baseline 下確實能形成可研究的 degradation，不是要求每個人複製同一種圖片。

| Failure family | 它在問什麼？ | 教師固定／提供 | 學生自行決定 | 必要 metrics | 文獻原型 |
| --- | --- | --- | --- | --- | --- |
| **Global rule／高階關係** | 每個局部看起來都合理，但多個部件合起來必須滿足同一條 rule。例如顏色、方向或位置之間有 2-way、3-way、4-way relation。關係階數越高，附近的 local evidence 越不足。 | matched-complexity 的 2／3／4-way relation、各階相同的 marginal balance，以及 4-way anchor。原論文使用 parity；本作業必須改成有連續 variation 的影像，不能退化成 binary tokens。 | 圖案、物件與 rule 的真實語意；condition 是否指出 rule family。 | **Primary:** rule validity。**Guards:** local fidelity、每個 attribute 的 marginal balance、novelty／memorization check。 | [When Rule Learning Breaks](https://openreview.net/pdf?id=LjqX8OhPPi) |
| **Multi-object composition** | 單一物件都會畫，不代表模型能同時畫對數量、屬性綁定與空間關係。要先決定你研究的是 counting、binding 還是 relation，不能三個一起改。 | 固定畫布、物件 vocabulary 與 dataset budget；提供 object-count 或 scene-complexity 的三個 levels 與 anchor。 | 物件種類、顏色、位置的語意；可以把圓形換成細胞、建物或桌面物件。 | **Primary:** count accuracy、attribute-binding accuracy 或 relation accuracy（三選一）。**Guards:** object validity、coverage，以及另外兩項 composition metrics。 | [When Do Diffusion Models Learn to Generate Multiple Objects?](https://arxiv.org/abs/2605.00273) |
| **Long-tailed modes** | Head mode 表現很好，可能同時掩蓋 tail mode 已經消失。真正的問題是相對 class／mode frequency，而不是整體 valid rate。 | 固定 modes、within-mode variation、總 training draws 與三組 imbalance ratios；提供一個已驗證會出現 tail degradation 的 anchor。 | 哪些圖案代表 head／tail，以及這種不平衡在真實應用中的來源。 | **Primary:** macro per-mode recall 與 tail recall／tail quality。**Guards:** overall validity、head quality、generated frequency calibration。不能只報 overall valid rate。 | [Improving Generation Quality of Long-Tailed Diffusion](https://proceedings.mlr.press/v322/rodriguez26a.html) |
| **Heavy-tailed extremes** | 大多數樣本在中心區域，但少數合法事件有非常大的亮度、尺寸、強度或幅度。這和「某一個離散 mode 很少」不同：tail 是同一個連續 distribution 的極端值。 | 以 unit central scale 的 Student-$t$ family 改變 tail parameter $\nu$；提供三個 $\nu$ 與 heavy-tail anchor，避免把 global scale 當成 failure。 | 極端值代表的真實意義，例如強降雨、熱點強度、物件尺寸或材料缺陷幅度。 | **Primary:** $q_{0.95}/q_{0.99}$ calibration、exceedance rate。**Guards:** median／central quantiles、central fidelity、sample maximum 的 uncertainty。 | [Heavy-Tailed Diffusion Models](https://arxiv.org/abs/2410.14171) |
| **Global topology** | Pixel-wise 看起來接近，不代表道路或血管仍連通、輪廓仍封閉，或洞與 connected components 的數量正確。這類 failure 必須用 topology metric 才看得到。 | 固定 resolution、stroke width 與 pixel mass；只讓 path length、hole count 或 dependency span 其中一項改變，並提供 topology anchor。 | 道路、血管、segmentation mask、封閉輪廓或細胞排列的語意。 | **Primary:** connectivity、component count、hole count／Betti number。**Guards:** pixel fidelity、foreground mass、local smoothness。 | [TopoCellGen](https://openaccess.thecvf.com/content/CVPR2025/html/Xu_TopoCellGen_Generating_Histopathology_Cell_Topology_with_a_Diffusion_Model_CVPR_2025_paper.html) |

### Paper、starter generator 與 anchor 各自負責什麼？

- **Paper** 告訴我們這個 failure family 不是憑空捏造的，也提供可追查的研究語境。
- **Starter generator** 把問題縮成這份作業能跑的 continuous tensor，固定容易混淆的 nuisance variables。
- **Calibrated range** 是課程在這份固定 `FlatVelocityNetwork` 與 training budget 下實測過的 easy／medium／hard。
- **Anchor setting** 是一個已知會出現 degradation 的參考點。學生可以改視覺語意，但不能把論文使用的不同 architecture 或 budget 當成自己的 calibration evidence。

一個 package 只有同時通過下面三件事，才可以標成 calibrated：

1. easy model 相對 truth-vs-truth reference 有可用的 baseline；
2. anchor 的 primary metric 在多個 seeds 都退化；
3. guard metrics 沒有指出 scale mismatch、空白圖、mode reassignment 或 metric shortcut。

學生真正需要自行完成的是：real problem 與 toy 的連結、failure mechanism、可推翻的預測、結果解釋與 diagnostic。你可以沿用 starter generator，也可以換 visual representation；只要數學骨架、primary metric 與 guard metric 仍然對得起來。

## Policy 組：可以發展的方向

| 方向 | 真實資料中可能代表什麼？ | Toy 裡可以控制的量 |
| --- | --- | --- |
| **Narrow success region** | 機器人必須穿過窄通道、精準接觸物體，或在很小的 tolerance 內到達目標。軌跡看起來只差一點，task success 卻可能完全不同。 | 固定 start、goal、horizon 與 action scale，只改 clearance 或 demonstration support 的寬度。Success threshold 本身不能是唯一改變的東西。 |
| **同一個 state 有多種策略** | 面對同一個 observation，向左或向右繞過障礙都可能成功；不同 demonstrators 也可能有不同但都合理的風格。這是 $p(X_1\mid C)$ 的 multimodality，不是 paired target prediction。 | 固定 state 與每條 route 的品質，只改可行策略的數量、route separation 或 mixture weights；一次只選其中一個。評估 route coverage 與 validity，不使用逐筆 trajectory MSE。 |
| **Demonstration imbalance** | 收集到的 demonstrations 大多使用某一種策略，另外一條合法路徑很少出現。模型可能只學到 dominant behavior。 | 固定 routes、state distribution 與 trajectory noise，只改策略的 mixture weight $\pi_k(\lambda)$。除了整體 success，也要報每條 route 的 recall。 |
| **不同 states 有不同難度** | 有些 start–goal pairs 幾乎是直線就能完成，有些則需要繞障礙、精準轉彎或更長規劃。只看平均 success 可能把困難 states 蓋掉。 | 先定義 state groups 與各自難度；讓 $\lambda$ 控制 hard-state 的比例，或只控制一個幾何難度。必須報 per-state-group metric。 |
| **Partial observation** | $C$ 沒有包含完整狀態，例如障礙物被遮住、目標意圖未知，或速度資訊缺失。同一個 observation 可能對應不同的正確 future actions。 | 從完整 latent state 產生 $(X_1,C)$，再明確寫出 $C$ 刪掉了什麼。要比較 full-state oracle、只用 $C$ 的 Bayes limit 與 Flow Matching。 |
| **較長時間的 dependency** | 很早的一個 action 會影響很晚才發生的成功或碰撞；局部每一步都合理，整段 action chunk 合起來才看得出問題。 | 保持 tensor horizon 固定，讓 $\lambda$ 控制 dependency span、delayed constraint 出現的位置，或需要協調的 time steps 數量。 |

## 選方向時，先做這個小檢查

一個合格的題目應該可以完成下面這句話：

> 在真實的 __________ 資料中，我可能看到 __________。我選擇 __________ failure family，並用 $\lambda$ 改變 __________ 來保留這個困難；同時固定 __________，所以 easy 到 hard 的差異主要可以歸因於同一個 data property。

不要只寫「我選 long tail」或「我選 partial observation」。這些只是現象的名字；你的任務是把它變成一個有對象、有機制，也有可控制參數的問題。


## 不接受的 failure

- 把 continuous output 換成 discrete token 或 class index；
- 把 Gaussian variation 完全拿掉，或把 within-mode variation 壓到幾乎是離散點；
- 只讓 $\lambda$ 改變 metric threshold，而沒有改變 $p_{\mathrm{toy},\lambda}$；
- 只把所有資料整體放大、縮小或改 contrast，讓固定 network 因數值尺度而壞掉；
- 故意使用錯誤 normalization、錯誤 shape 或錯誤 condition pairing；
- 讓 `sample_target` 依賴 batch size、呼叫次數、training/evaluation mode 或其他 hidden global state；
- 修改 path、training target、loss 或正式 sampler；
- 減少 training steps、network width 或 NFE；
- 只換一個比較差的 random seed；
- 同時改很多 data properties，最後無法知道是哪一項造成差異；
- 要求模型使用從來沒有放進 $C$ 的資訊，卻不承認這是 partial observability；
- 直接把 $X_1$ 或它幾乎完整的 noisy copy 塞進 $C$，把 conditional generation 偷換成 denoising task，卻沒有真實世界理由與 Bayes/reference analysis。

你的 $X_1$ 必須保持為 continuous real-valued tensor。三個 difficulty levels 必須使用相同物理單位與 normalization；若全域 mean 或 standard deviation 明顯隨 $\lambda$ 改變，你必須再做 standardized control，證明現象不只是 scale mismatch。

Partial observation 可以研究，但必須把「模型能否表示 $p(X_1\mid C)$」和「只知道 $C$ 時，任何方法最多能做到什麼」分開。後者要用 Bayes-optimal risk、oracle simulation，或 real-vs-real reference 定出不可約誤差；不可約誤差不能算成 Flow Matching failure。


## 1.1 真實世界原型

請先回答以下問題，再寫 toy generator：

1. 你選擇影像還是 trajectory policy？
2. 真實世界中，什麼資料會具有你選的 property？
3. 為什麼這個 property 可能讓 generative model 變得困難？
4. 請放一張 real-data figure、資料範例或自己畫的情境圖。

### 你的回答

**領域：** `[請填寫]`

**真實情境：**

`[請在這裡作答並插入圖片]`

**你想保留的 data property：**

`[請在這裡作答]`


## 1.2 Real problem 怎麼縮成 toy？

| 問題 | 你的設計 |
| --- | --- |
| 真實世界的一筆資料是什麼？ | `[請填寫]` |
| 你選擇哪個 image failure family／policy design pattern？ | `[請填寫]` |
| 參考哪一篇 paper 或哪個真實資料來源？ | `[請填寫並附連結]` |
| Toy 裡的 $X_1$ 是什麼？ | `[請填寫]` |
| Condition $C$ 是什麼？ | `[請填寫]` |
| Toy 保留哪一個困難？ | `[請填寫]` |
| Package 規定的 primary metric 是什麼？ | `[請填寫]` |
| 哪一個 guard metric 可以抓到假解？ | `[請填寫]` |
| Toy 刪除了哪些真實複雜度？ | `[請填寫]` |
| 為什麼刪除後仍能回答你的問題？ | `[請填寫]` |

只能有一個 primary change。請用一句話完成：

> **相較於 easy toy，我只改變 __________；其他資料設定保持不變。**

若你更換 starter generator 的 visual representation，再補一句：

> **我保留 package 的 __________ 數學骨架；把 __________ 換成 __________，但 primary metric 與 guard metric 仍分別測量 __________ 與 __________。**


## 1.3 用數學定義你的資料

如果是 conditional problem，請寫出

$$
(X_1,C)\sim p_{\mathrm{toy},\lambda}(x_1,c).
$$

如果是 unconditional problem，請寫出

$$
X_1\sim p_{\mathrm{toy},\lambda}(x_1),
\qquad C=\varnothing.
$$

$\lambda$ 是你用來控制困難程度的參數。至少定義三個值：

$$
\lambda_{\mathrm{easy}},
\qquad
\lambda_{\mathrm{medium}},
\qquad
\lambda_{\mathrm{hard}}.
$$

這三個值必須真的產生三個不同的資料分布，而不是只改 evaluation threshold。請完整說明 sampling procedure，並明確指出 $\lambda$ 出現在哪一個 probability、mean、variance、geometry 或 conditional relation 裡；不要只寫「從我的 dataset 抽一筆」。

同時寫出你要固定的 nuisance properties。尤其要回答：

- 三個 levels 的全域 mean、standard deviation 與 normalization 是否相近？
- 若有 rare mode，$N\rho$ 預期會看到幾筆？
- 若有 mixture modes，within-mode variation 是否在三個 levels 保持不變？
- 若有 partial observation，給定 $C$ 的不可約風險或 truth ceiling 是多少？

### 你的數學定義

`[請在這裡寫下 distribution、sampling procedure、三個 difficulty levels，以及保持不變的 nuisance properties]`


## 1.4 Symbol-to-code table

| 數學符號 | 在你的問題裡代表什麼？ | Tensor shape | 程式變數 |
| --- | --- | --- | --- |
| $X_1$ | `[請填寫]` | `[請填寫]` | `x1` |
| $C$ | `[請填寫]` | `[請填寫]` | `c` |
| $X_0$ | Gaussian source | 與 $X_1$ 相同 | `x0` |
| $I_t$ | `[請填寫]` | `[請填寫]` | `xt` |
| $U_t$ | `[請填寫]` | `[請填寫]` | `target_velocity` |
| $\lambda$ | `[請填寫]` | scalar | `difficulty` |

接著用一句話解釋：為什麼 $X_1$ 與 $C$ 必須一起抽，而 $X_0$ 可以獨立抽？

`[請在這裡作答]`


## 1.5 實作你的 toy data generator

完成下面三個設定與 `sample_target`。它必須回傳：

- `x1`：shape `(batch, *DATA_SHAPE)`；
- `c`：shape `(batch, COND_DIM)`，或 unconditional 時回傳 `None`。

同一個 `difficulty` 必須永遠代表同一個 $p_{\mathrm{toy},\lambda}$。函式只可以依賴傳入的 arguments 與 random-number generator；不能查看 batch size 來切換規則，也不能使用 call counter 或 training/evaluation flag。Easy、medium、hard 只改你在 1.3 寫下的 primary property。

等一下的 audit 會用相同 seed 比較三個 levels。若 $X_1$ 與 $C$ 都逐元素完全相同，代表你很可能只改了 metric，而沒有改資料。


In [ ]:
# TODO: choose exactly one track.
TRACK = "image"  # "image" or "trajectory"

# Required only for the image track. Choose one calibrated course package.
IMAGE_FAILURE_FAMILY = "global_rule"  # global_rule, multi_object, long_tail, heavy_tail, topology

# TODO: replace these with your data shapes.
DATA_SHAPE = (1, 8, 8)
COND_DIM = 0

# For the image track, start from the three values supplied by your package.
# You may change visual semantics, but do not invent an uncalibrated range and
# then call it a package anchor.
DIFFICULTY_LEVELS = {
    "easy": 0.0,
    "medium": 0.5,
    "hard": 1.0,
}

def sample_target(batch_size, difficulty, device=DEVICE):
    # Return paired samples (x1, c) from p_toy,difficulty.
    # difficulty MUST change the data distribution, not only a metric threshold.
    # Do not branch on batch_size, call count, or train/eval mode.
    # x1: (batch_size, *DATA_SHAPE), float32
    # c:  (batch_size, COND_DIM), float32, or None
    # TODO: implement or adapt your package's continuous toy distribution.
    raise NotImplementedError


## 1.6 先只看資料，不要先看 model results

對 easy、medium、hard 各畫至少 16 筆 $X_1$。如果是 conditional problem，圖上也要標出對應的 $C$。

看完後回答：

- $\lambda$ 增加時，肉眼可以看到什麼變化？
- 哪些東西刻意保持相同？
- 這三組資料真的只差一個 primary property 嗎？


In [ ]:
# TODO: visualize at least 16 data samples from each difficulty level.
# Your figure should make the controlled change visible without showing model outputs.

fig = None
# ... your visualization code ...


### 你從資料圖讀到什麼？

`[請在這裡作答。若資料圖已經和你原先想像不同，現在就要說明。]`


---

# 問題二：為什麼你覺得固定的 Flow Matching 會做不好？

先別跑正式訓練。

你現在已經定義好資料。接著要先留下自己的 failure intuition：不是「因為 hard 比較難」，而是具體指出 **velocity network 在什麼地方要同時處理哪些互相衝突、稀少或精細的訊號**。


## 2.1 先用直覺解釋

請回答：

1. $\lambda$ 從 easy 走向 hard 時，資料幾何或 conditional structure 發生什麼變化？
2. 對相近的 $(I_t,t,C)$，可能出現哪些不同的 target velocities？
3. 這些 velocities 是方向接近、互相抵消，還是只有少數樣本才看得到？
4. 你預期 generated samples 會怎麼壞？
5. 哪一群 samples、哪一種 condition，或哪一段 $t$ 應該最先壞？

請加一張自己畫的示意圖。圖中至少標出 $X_0$、$X_1$、$I_t$ 與兩個可能的 velocity targets。

### 你的直覺

`[請在這裡作答並插入示意圖]`


## 2.2 用一條式子把直覺釘住

Squared-error regression 的最佳解是

$$
v^\star(x,t,c)
=
\mathbb E[X_1-X_0\mid I_t=x,C=c].
$$

請用你的 toy 回答：

- 哪些不同的 $(X_0,X_1)$ 可能產生相近的 $(I_t,C)$？
- 它們的 $X_1-X_0$ 如何不同？
- 取 conditional mean 後，哪一種資訊可能被保留、變弱或抵消？

不要求完整求出 $v^\star$，但你的直覺必須能和這個 conditional expectation 對上。

### 你的回答

`[請在這裡作答]`


## 2.3 寫下可被推翻的預測

這一節必須放在 results 前面。即使實驗結果不同，也不要刪掉或改寫成事後看來正確的版本。

| 問題 | 實驗前的預測 |
| --- | --- |
| 你定義的 failure 是什麼？ | `[請填寫 quantitative criterion]` |
| 哪個 metric 會最先變差？ | `[請填寫]` |
| easy → medium → hard 應該是漸進還是突然惡化？ | `[請填寫]` |
| 哪類 samples／conditions 最先失敗？ | `[請填寫]` |
| 什麼結果會推翻你的 mechanism？ | `[請填寫]` |

再用一個完整句子押注：

> **我預測當 $\lambda$ 從 ______ 增加到 ______ 時，______ 會先發生，因為 ______。**


## 2.4 定義 metrics 與一個 useful value

至少使用一個原有 task metric。影像組的 primary metric 與 guard metric 由所選 package 規定，兩者都必須報告；你可以再增加自己的 metric，但不能用它取代 package metrics：

- 影像：valid rate、mode coverage、per-mode recall、nearest-template distance；
- 軌跡：collision rate、success rate、route coverage、goal error。

但在寫 code 前，先處理三個很容易誤判的地方。

### A. Generated sample 和 reference 不是一一配對

即使它們使用相同的 $C$，兩筆 $X_1\sim p(X_1\mid C)$ 仍可能是獨立而且不同的合法答案。因此，除非你已證明每個 $C$ 對應唯一 target，否則不能把

$$
\frac1N\sum_i\|\hat X_{1,i}-X_{1,i}^{\mathrm{ref}}\|
$$

當成主要 metric。請使用 distribution-level、validity、coverage 或 per-group metric。`reference_a` 與 `reference_b` 會是兩組獨立 truth samples，讓你用同一套 metric 算 **truth-vs-truth floor/ceiling**。

### B. Metric 必須看得到少數群體

除了 aggregate metric，至少報告一個 per-mode／per-route／per-condition-group metric。分類結果必須容許 `invalid` 或 `neither`；不能強迫每個 sample 都被算進最近的合法 mode。

若使用 rare-mode recall，正式 evaluation 必須滿足預期 rare count $N_{\mathrm{eval}}\rho\ge 50$，或報告 binomial confidence interval／exact tail probability。這種情況可以只增加 evaluation samples，不算修改 training budget。

### C. 先定義 reference，再談 failure

對每個主要 metric 都要先在兩組真實資料上跑一次，估計 metric 本身的 sampling floor/ceiling。若研究 partial observation，再寫出只給 $C$ 時的 Bayes-optimal risk、oracle performance 或可計算的 lower bound。只有超過這個不可約部分的 gap，才可能是 model failure。

另外，自行定義一個直接服務於 explanation 的 useful value，例如：

- rare mode 在 training 中預期出現的次數；
- mode separation／within-mode standard deviation；
- conditional mean image 到最近合法模板的距離；
- mean action trajectory 到障礙物的最小距離；
- 相近 $(I_t,C)$ 的 target-velocity disagreement；
- 各 time bins 的 velocity error。

先寫公式、說明它為什麼有用，再寫 code。不要先印出一個容易算的數字，才替它補理由。

### 你的 metric、reference 與 useful value

`[請在這裡寫公式、better/worse 方向、truth-vs-truth reference、per-group metric、failure threshold 與 useful value]`


In [ ]:
def evaluate_samples(
    generated,
    reference_a,
    condition_a,
    reference_b,
    condition_b,
    difficulty,
):
    # generated was sampled using condition_a, but generated[i] and reference_a[i]
    # are NOT paired targets. Use distribution/validity/group metrics unless the
    # target is mathematically unique given C.
    #
    # Return scalar metrics for BOTH:
    #   1) model vs truth, and
    #   2) truth A vs truth B (the metric floor/ceiling).
    # Include an explicit invalid/neither group when classifying samples.
    # TODO: implement your metrics.
    raise NotImplementedError


def compute_useful_value(difficulty):
    # Compute the diagnostic quantity defined in your report.
    # TODO: this may use data samples without training a model.
    raise NotImplementedError


---

# 3. 正式實驗

現在才開始訓練。

Easy、medium、hard 必須使用完全相同的 model capacity、training budget、sampler 與 NFE。三個正式 training seeds 也必須相同，這樣才是對 training randomness 的 paired comparison。若 rare-mode metric 需要較多 evaluation samples，可以增加 **所有 levels 的 evaluation N**，或另外做一個不重訓模型的 evaluation-only analysis，並清楚標示。

下面的 experiment runner 不需要知道你選影像還是軌跡。它只依賴前面定義的 `DATA_SHAPE`、`COND_DIM`、`sample_target` 與 `evaluate_samples`。

Failure 的主要量應該是 model 相對於該 level 的 truth reference 所多出的 gap，再比較這個 gap 從 easy 到 hard 如何改變。不要只看 hard 的 raw score，因為資料本身的 Bayes difficulty 與 metric sampling noise 也可能一起改變。


In [ ]:
def checked_target(batch_size, difficulty):
    x1, c = sample_target(batch_size, difficulty, device=DEVICE)
    x1 = x1.to(device=DEVICE, dtype=torch.float32)
    assert tuple(x1.shape) == (batch_size, *DATA_SHAPE)
    assert torch.isfinite(x1).all()
    if COND_DIM == 0:
        assert c is None
    else:
        assert c is not None
        c = c.to(device=DEVICE, dtype=torch.float32)
        assert tuple(c.shape) == (batch_size, COND_DIM)
        assert torch.isfinite(c).all()
    return x1, c


def train_one(difficulty, seed, steps=TRAIN_STEPS, verbose=True):
    set_seed(seed)
    velocity = FlatVelocityNetwork(
        sample_shape=DATA_SHAPE,
        cond_dim=COND_DIM,
        hidden=HIDDEN,
        layers=LAYERS,
    ).to(DEVICE)
    flow = FlowMatching(velocity, lr=LEARNING_RATE)

    def get_batch():
        x1, c = checked_target(BATCH_SIZE, difficulty)
        x0 = torch.randn_like(x1)
        return x0, x1, c

    history = flow.fit(
        get_batch,
        steps=steps,
        name=f"lambda={difficulty}, seed={seed}",
        verbose=verbose,
    )
    return flow, history


@torch.no_grad()
def generate_and_evaluate(flow, difficulty, seed, n=N_EVAL, sample_steps=SAMPLE_STEPS):
    set_seed(seed + 10_000)
    reference_a, condition_a = checked_target(n, difficulty)
    x0 = torch.randn_like(reference_a)
    generated = flow.sample(x0, c=condition_a, steps=sample_steps)

    # A second independent truth set is used to calibrate metric noise/floor.
    set_seed(seed + 20_000)
    reference_b, condition_b = checked_target(n, difficulty)
    metrics = evaluate_samples(
        generated,
        reference_a,
        condition_a,
        reference_b,
        condition_b,
        difficulty,
    )
    to_cpu = lambda x: None if x is None else x.cpu()
    return (
        generated.cpu(),
        reference_a.cpu(),
        to_cpu(condition_a),
        reference_b.cpu(),
        to_cpu(condition_b),
        metrics,
    )


## 3.1 Data-generator audit：先證明你真的只改了一件事

先執行下面的 audit，再訓練模型。它會檢查 shape、finite values、同 seed 可重現性，並列出每個 level 的 basic statistics。它也會用相同 random seed 檢查 easy 與 hard 是否逐元素完全相同；這只能抓到最明顯的漏洞，不能代替你的數學說明。

把輸出整理成一張表，並回答：

1. 哪個統計量顯示 primary property 確實隨 $\lambda$ 改變？
2. 哪些 nuisance statistics 大致保持固定？
3. 若 mean/std 改變超過約 10%，這是現象本身不可分割的一部分嗎？standardized control 的結果如何？
4. 有沒有 mode／route 因為位置、邊界或 sample count，而同時變得更難？若有，請 counterbalance 或承認這是 confound。

`[請在 audit output 下方回答]`


In [ ]:
@torch.no_grad()
def audit_data_generator(n=4096, seed=31415):
    rows = []
    snapshots = {}

    for level_name, difficulty in DIFFICULTY_LEVELS.items():
        set_seed(seed)
        x1, c = checked_target(n, difficulty)
        flat_x = x1.reshape(n, -1)
        joined = flat_x if c is None else torch.cat([flat_x, c.reshape(n, -1)], dim=1)
        snapshots[level_name] = joined.detach().cpu()

        row = {
            "level": level_name,
            "lambda": difficulty,
            "x_mean": flat_x.mean().item(),
            "x_std": flat_x.std().item(),
            "x_min": flat_x.min().item(),
            "x_max": flat_x.max().item(),
        }
        if c is not None:
            row.update(c_mean=c.mean().item(), c_std=c.std().item())
        rows.append(row)

        # A generator should be reproducible when the RNG is reset.
        set_seed(seed + 1)
        xa, ca = checked_target(256, difficulty)
        set_seed(seed + 1)
        xb, cb = checked_target(256, difficulty)
        assert torch.equal(xa, xb), f"{level_name}: sample_target is not seed-reproducible"
        if ca is not None:
            assert torch.equal(ca, cb), f"{level_name}: condition generator is stateful"

    if torch.equal(snapshots["easy"], snapshots["hard"]):
        raise AssertionError(
            "easy and hard data are identical under the same random draws. "
            "Did difficulty only change the metric?"
        )

    for row in rows:
        print(row)
    return rows


# Run this before any model training.
# data_audit_rows = audit_data_generator()


## 3.2 Smoke test

先用 20 training steps、32 evaluation samples 檢查 shapes 與 code path。這裡的數字沒有科學意義，不得進入正式結論。


In [ ]:
SMOKE_DIFFICULTY = DIFFICULTY_LEVELS["easy"]

# Uncomment after sample_target() and evaluate_samples() are implemented.
# smoke_flow, _ = train_one(SMOKE_DIFFICULTY, seed=0, steps=20)
# (smoke_generated, smoke_reference_a, smoke_c_a,
#  smoke_reference_b, smoke_c_b, smoke_metrics) = generate_and_evaluate(
#     smoke_flow, SMOKE_DIFFICULTY, seed=0, n=32, sample_steps=2
# )
# print(smoke_metrics)


## 3.3 三個 difficulty levels × 三個 paired seeds

這會訓練九個 models。確認下面三件事後，才把 `RUN_FINAL` 改成 `True`：

1. data-generator audit 通過，而且三個 levels 的資料真的不同；
2. 你已經在 2.3 寫下 prediction 與可推翻的 threshold；
3. easy-baseline gate 通過：easy model 相對 truth reference 的 gap 足夠小，讓後面的 degradation 有意義。

這些都在同一份 notebook 完成，不需要中途繳交或額外 checkpoint。


In [ ]:
RUN_FINAL = False

records = []
example_outputs = {}

if RUN_FINAL:
    for level_name, difficulty in DIFFICULTY_LEVELS.items():
        print(f"\n===== {level_name}: lambda={difficulty} =====")
        print("useful value:", compute_useful_value(difficulty))
        for seed in FINAL_SEEDS:
            flow, history = train_one(difficulty, seed=seed)
            (generated, reference_a, condition_a,
             reference_b, condition_b, metrics) = generate_and_evaluate(
                flow, difficulty, seed=seed
            )
            record = {"level": level_name, "difficulty": difficulty, "seed": seed, **metrics}
            records.append(record)
            print(record)
            if seed == FINAL_SEEDS[0]:
                example_outputs[level_name] = {
                    "generated": generated,
                    "reference_a": reference_a,
                    "condition_a": condition_a,
                    "reference_b": reference_b,
                    "condition_b": condition_b,
                    "history": history,
                }
            del flow
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


## 3.4 Summary table 與主圖

Summary table 至少要有每個 metric 的 mean ± standard deviation，並把 **model score、truth-vs-truth reference、兩者的 gap** 分開。主圖必須讓讀者同時看到：

1. easy／medium／hard 的資料差異；
2. easy／medium／hard 的生成結果；
3. 至少一個 aggregate metric 與一個 per-group metric 隨 $\lambda$ 的變化；
4. truth-vs-truth floor／ceiling；
5. 你定義的 useful value。

圖不能只放最好看的 seed。範例視覺可以使用固定 seed，但數字必須來自三個 seeds。若某一類別在 evaluation set 中太少，要報告 count 與不確定性，不能只報一個百分比。


In [ ]:
# TODO: aggregate records into mean ± std and create the summary figure.
# You may use pandas if you want, but it is not required.

# ... summary table ...
# ... one summary figure ...


### Results：圖和數字實際說了什麼？

先描述 observation，不要在第一句就宣稱原因。

`[請在這裡放 summary figure、summary table，並描述最重要的兩到三個 observations。]`


---

# 4. 結果和原本想的一樣嗎？

回到 2.3 的押注，逐項比較。不要只寫「符合預期」或「不符合預期」。

| 原本的預測 | 實際結果 | 符合程度 | 支持你的證據 |
| --- | --- | --- | --- |
| `[請填寫]` | `[請填寫]` | 符合／部分符合／不符合 | `[figure、metric 或 useful value]` |
| `[請填寫]` | `[請填寫]` | 符合／部分符合／不符合 | `[figure、metric 或 useful value]` |


## 4.1 如果結果符合預期

請回答：

- 哪一張圖或哪一個數字最直接支持原本的 failure mechanism？
- $v^\star(x,t,c)=\mathbb E[X_1-X_0\mid I_t=x,C=c]$ 如何幫助解釋？
- model-to-truth 與 truth-to-truth 的 gap 是否隨 $\lambda$ 增加？
- 你的 useful value 能不能預測 failure 發生的位置或強度？
- standardized control、per-group metric 或 Bayes/reference analysis 是否排除了更簡單的解釋？
- 是否還有另一個 mechanism 也可能產生相同結果？

## 4.2 如果結果不符合預期

請回答：

- 模型真的沒有失敗，還是 metric 沒有抓到 failure？
- Toy 是否沒有保留你以為的真實困難？
- Flow Matching 是否用你沒有預期到的方式保留了 distribution？
- 哪一個額外計算可以區分這些解釋？

若原始設計沒有出現可量測的 degradation，你**必須**追加一個 revised harder setting；保留原本 negative result，再只把同一個 primary property 往前推。不能改成離散資料、scale trick 或同時加入第二個困難。

### 你的解釋

`[請依照符合／不符合的情況作答。請把 intuition、公式、圖、truth reference、per-group metric 與 useful value 接成同一條推理。]`


## 4.3 一個必要的 diagnostic experiment

最後**必須**做一個診斷實驗。目的不是把結果救到最好，而是區分原因。只能改一件事，例如：

- hard setting 增加 NFE；
- hard setting 增加 training steps；
- hard setting 暫時增加 network width；
- partial-observation toy 補回缺少的 condition；
- imbalanced toy 暫時重新平衡資料；
- 對三個 levels 使用同一個 standardization，檢查是否只是 scale mismatch。

Diagnostic 不算正式方法，因此必須和前面的固定設定分開報告。先寫下：**如果你的 mechanism 是對的，這個 diagnostic 應該改變什麼？如果它不改變，又會排除什麼？**

最後把 evidence 分到下列其中一類；不要只寫「有改善」或「沒改善」。

1. **Capacity／optimization-limited**：增加 width 或 training budget 明顯縮小 gap。
2. **Sampler-limited**：增加 NFE 明顯縮小 gap。
3. **Irreducible／observation-limited**：truth/Bayes reference 本身也有同等限制。
4. **Persistent data-model mismatch**：在上述 control 後，hard gap 仍存在。
5. **仍無法區分**：目前 diagnostic 不足；清楚寫出還缺什麼 evidence。

`[請先作答，再執行下面的實驗]`


In [ ]:
# TODO: run exactly one diagnostic intervention on the hard setting.
# Keep the original hard-setting result. Do not replace it with this diagnostic result.

# ... diagnostic experiment ...


### Diagnostic 結果

| 原始 hard gap | Diagnostic 後的 gap | 分類 | 這個結果排除了什麼？ |
| ---: | ---: | --- | --- |
| `[填寫]` | `[填寫]` | capacity / sampler / irreducible / persistent / unresolved | `[填寫]` |

`[它支持、削弱，還是無法區分你的 explanation？請引用數字與 reference 作答。]`


---

# 5. AI／research decision log

你可以使用 AI 協助 coding、debug、公式檢查、視覺化與文字整理。評分不看 AI 用得多或少，而是看你有沒有驗證建議，以及能不能解釋最後留下來的內容。

如果沒有使用 AI，也請記錄三個自己或與同學討論後做出的重要研究決策。

| 你當時遇到的問題 | 建議來自 AI／同學／自己 | 得到的建議 | 你如何驗證？ | 接受或拒絕？為什麼？ |
| --- | --- | --- | --- | --- |
| `[請填寫]` |  |  |  |  |
| `[請填寫]` |  |  |  |  |
| `[請填寫]` |  |  |  |  |

最後請列出哪些 code cells 曾受到 AI 協助，以及你實際檢查了什麼。不要只寫「AI 幫我 debug」。


# 6. 最後回到開頭

請用 150–250 字回答：

> **你的實驗告訴我們，固定的 Flow Matching framework 面對哪一種真實 data property 時會開始遇到困難？目前證據支持的原因是什麼？哪些部分仍然不能確定？**

`[請在這裡寫最後結論]`


## 繳交前 checklist

- [ ] 我的 toy 來自一個說得清楚的真實資料現象。
- [ ] 若選影像，我已指出 failure family、文獻原型、package primary metric 與 guard metric。
- [ ] 我沒有把 paper 中不同 architecture／budget 的結果冒充成自己 toy 的 calibrated anchor。
- [ ] $X_1$ 仍是 continuous real-valued tensor，且沒有用近乎離散的 variance 偷做 failure。
- [ ] Easy、medium、hard 只改一個 primary property，而且 $\lambda$ 真的改變 $p_{\mathrm{toy},\lambda}$。
- [ ] `sample_target` 不依賴 batch size、call count、train/eval mode 或 hidden global state。
- [ ] 三個 levels 使用相同物理單位與 normalization；若 scale 改變，已完成 standardized control。
- [ ] 我沒有修改 fixed Flow Matching path、target、loss 或正式 sampler。
- [ ] 我定義了 $p_{\mathrm{toy},\lambda}$、所有 symbols、tensor shapes 與 nuisance properties。
- [ ] 我畫出資料並報告每個 level 的 mean、std、range 與 relevant group counts。
- [ ] Failure intuition 寫在 results 前面，而且有 quantitative、可被推翻的 prediction。
- [ ] Easy baseline 相對 truth reference 足夠好，正式結果不是三個 levels 一起壞。
- [ ] 正式結果使用三個 paired training seeds。
- [ ] 每個主要 metric 都有 truth-vs-truth floor／ceiling，且沒有誤用 paired sample distance。
- [ ] 我同時報告 aggregate 與 per-mode／per-route／per-group metric，分類允許 invalid／neither。
- [ ] Rare-mode evaluation 有足夠 expected count，或附上 binomial uncertainty。
- [ ] Partial-observation 題目有 Bayes/oracle reference，且沒有把不可約誤差叫做 model failure。
- [ ] 我報告了 task metric 與自己定義的 useful value。
- [ ] 若第一次沒有 measurable degradation，我保留 negative result 並追加一個合理的 harder setting。
- [ ] 我完成一個 diagnostic，並分類 failure source。
- [ ] Explanation 同時引用 intuition、公式、圖、truth reference 與數字。
- [ ] Notebook 可以 restart and run all。
- [ ] 我完成 AI／research decision log。
- [ ] 我已輸出 PDF 或 HTML，確認公式、圖與表格沒有被切掉。
